In [1]:
%load_ext autoreload
%autoreload 2

## Initial Test

### Draw

Make Chip

In [1]:
from qiskit_metal.quantrolib.chip import JAWS

chip = JAWS()

In [2]:
chip.draw()

Place Resonators

In [ ]:
3189 - 680/2 - 25

In [8]:
from qiskit_metal.quantrolib.resonator import IncaResonatorShortedMasked

resonator = IncaResonatorShortedMasked(
    design=chip,
    options=dict(center_offset=1),
)
resonator = IncaResonatorShortedMasked(
    design=chip,
    options=dict(center_offset=0),
)
resonator = IncaResonatorShortedMasked(
    design=chip,
    options=dict(center_offset=-1),
)

# resonator = Resonator(
#     design=chip,
#     options=dict(pos_x="2824um", pos_y="0um", orientation="0"),
# )


In [ ]:
chip.draw()

Add DC Probes

In [ ]:
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed


DC_probe_1 = RouteFramed(chip._design, 'DC_probe_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='rf_port_2', pin='out'),
        end_pin=dict(component='resonator_1', pin='DC_probe'),
    ),
    lead=dict(end_straight='2mm', start_straight='0.4mm'),
))

# DC_probe_2 = RouteFramed(chip._design, 'DC_probe_2', options=dict(
#     pin_inputs=dict(
#         start_pin=dict(component='rf_port_3', pin='rf_port_3'),
#         end_pin=dict(component='resonator_2', pin='DC_probe'),
#     ),
#     lead=dict(end_straight='2mm', start_straight='0.4mm'),
# ))


In [ ]:
chip.draw()

In [9]:
transmission_1 = RouteFramed(chip._design, 'transmission_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='rf_port_0', pin='out'),
        end_pin=dict(component='rf_port_1', pin='out'),
    ),
))

# transmission_2 = RouteFramed(chip._design, 'transmission_2', options=dict(
#     pin_inputs=dict(
#         start_pin=dict(component='rf_port_5', pin='rf_port_5'),
#         end_pin=dict(component='rf_port_4', pin='rf_port_4'),
#     ),
# ))

In [10]:
chip.draw()

Generate GDS

In [ ]:
chip.generate_gds("test_chip")


### Eigenmode Simulation

In [6]:
design = chip._design

#### Edit params of specific Components

In [7]:
import numpy as np

wire_width = 2  # um
wire_length = 94  # um
real_wire_width = 0.6 # um

kinetic_inductance_square = 2.e-13  # H/square
nanowire_inductance = kinetic_inductance_square * wire_width/real_wire_width * wire_length

design.components["resonator_1"].options.hfss_inductance = f"{nanowire_inductance*1e12} pH"


In [ ]:
design.components["resonator_1"].options

In [9]:
design.rebuild()

#### Simulation Setup

In [10]:
from qiskit_metal.renderers.renderer_ansys.hfss_renderer import QHFSSRenderer
fourq_hfss : QHFSSRenderer = design.renderers.hfss

In [ ]:
fourq_hfss.start()

In [ ]:
fourq_hfss.connect_ansys_design()

In [ ]:
fourq_hfss.new_ansys_design("HFSSMetalEigenmode", 'eigenmode')

fourq_hfss.connect_ansys_design("HFSSMetalEigenmode")

Initialize

All components in the design:

In [ ]:
tuple(chip._design.components)

In [13]:
port_list = [(f'rf_port_{i}', 'in', 50) for i in range(6)]

In [ ]:
port_list

#### Run the Simulation

In [ ]:
fourq_hfss.options["max_mesh_length_jj"] = "0.5um"
fourq_hfss.options["max_mesh_length_port"] = "250um"
fourq_hfss.options

Rebuild to verfiy changes?

In [16]:
fourq_hfss.clean_active_design()
fourq_hfss.render_design(
    open_pins=[],
    port_list=port_list,
)

In [ ]:
fourq_hfss.initialize_eigenmode(
    name='Setup',
    min_freq_ghz=4,
    n_modes=3,
    max_delta_f=0.1,
    max_passes=10,
)

In [ ]:
fourq_hfss.activate_ansys_setup('Setup')

In [ ]:
fourq_hfss.analyze_setup('Setup')

In [ ]:
create_report = fourq_hfss.plot_fields(
    object_name="main",
    QuantityName='Mag_E',
)

In [ ]:
type(create_report)

In [ ]:
fourq_hfss.get_unique_component_ids()

## Framework

In [2]:
from qiskit_metal.quantrolib.simulation import ANSYS, RenderConfig, SimulationConfig, ReportConfig

### Configs

## Simple Chiplet

### Design

In [3]:
from qiskit_metal.quantrolib.chip import Chiplet
from qiskit_metal.quantrolib.resonator import VariableIncaResonator, EdgeInductanceResonator, IncaResonator, BraggResonator

def build_chiplet(resonator_type: str) -> Chiplet:
    chip = Chiplet(size_x="2mm", size_y="2mm")

    if resonator_type == "VariableIncaResonator":
        resonator = VariableIncaResonator(
            name="resonator",
            design=chip,
            options=dict(center_offset=0.0),
        )
    
    elif resonator_type == "EdgeInductanceResonator":

        resonator = EdgeInductanceResonator(
            name="resonator",
            design=chip,
            # options=dict(pos_x="2824um", pos_y="0um", orientation="0"),
        )
    elif resonator_type == "IncaResonator":
        resonator = IncaResonator(
            name="resonator",
            design=chip,
            # options=dict(pos_x="2824um", pos_y="0um", orientation="0"),
        )

    elif resonator_type == "BraggResonator":
        resonator = BraggResonator(
            name="resonator",
            design=chip,
            options=dict(n_pairs=18),
        )


    else:
        raise ValueError(f"Unknown resonator type '{resonator_type}'.")

    return chip

In [4]:
def build_example_chiplet() -> Chiplet:
    chip = Chiplet(size_x="2mm", size_y="2mm")

    for index, resonator_type in enumerate([
        IncaResonator,
        BraggResonator,
        VariableIncaResonator,
        EdgeInductanceResonator,
    ]):
        resonator = resonator_type(
            name=f"resonator{index}",
            design=chip,
            options=dict(pos_x=f"{index*2000}um"),
        )
    return chip

### Draw

In [5]:
chip = build_example_chiplet()

In [6]:
chip.draw()

In [ ]:
chip.generate_gds("test_chip")

### Post-process design

In [7]:
design = chip._design

In [ ]:
tuple(design.components)

In [9]:
wire_width = 2  # um
wire_length = 94  # um
real_wire_width = 0.6 # um

kinetic_inductance_square = 2.e-13  # H/square
nanowire_inductance = kinetic_inductance_square * wire_width/real_wire_width * wire_length

design.components["resonator"].options.hfss_inductance = f"{nanowire_inductance*1e12} pH"

### Simulate

In [ ]:
design.rebuild()

In [ ]:
render_config = RenderConfig(
    name="Render",
    design=design,
    design_name="HFSSMetalEigenmode",
    open_pins=[],
    port_list=[],
    max_mesh_length_jj="0.5um",
    max_mesh_length_port="250um",
)

simulation_config = SimulationConfig(
    name="Setup_bis",
    min_freq_ghz=4.0,
    n_modes=3,
    max_delta_f=0.1,
    max_passes=10,
)

report_config = ReportConfig(
    name="Fields",
    field_configs=[{"object_name": "main", "QuantityName": 'Mag_E', 'PlotFolder': 'E Field'}, {"object_name": "main", "QuantityName": 'Mag_H', 'PlotFolder': 'H Field'}]
)

ansys = ANSYS(configs=[render_config, simulation_config, report_config], run_upon_init=True)

### All in one

In [ ]:
from qiskit_metal.quantrolib.chip import Chiplet
from qiskit_metal.quantrolib.resonator import IncaResonator
from qiskit_metal.quantrolib.simulation import ANSYS, RenderConfig, SimulationConfig, ReportConfig

for index in range(1):

    # built
    n_pairs = 12 + index
    chip = Chiplet(size_x="2mm", size_y="2mm")
    
    resonator = IncaResonator(
        name=f"resonator_{n_pairs}",
        design=chip,
        options=dict(n_pairs=n_pairs),
    )

    # modification
    design = chip._design

    wire_width = 2  # um
    wire_length = 94  # um
    real_wire_width = 0.6 # um

    kinetic_inductance_square = 2.e-13  # H/square
    nanowire_inductance = kinetic_inductance_square * wire_width/real_wire_width * wire_length

    design.components[f"resonator_{n_pairs}"].options.hfss_inductance = f"{nanowire_inductance*1e12} pH"

    design.rebuild()

    # simulation
    render_config = RenderConfig(
        name="Render",
        project_dir="C:\\Users\\NT280827\\Documents\\Ansoft",
        project_name="Project12",
        design=design,
        design_name=f"HFSSMetalEigenmode_{n_pairs}",
        setups=[],
        open_pins=[],
        port_list=[],
        max_mesh_length_jj="0.5um",
        max_mesh_length_port="250um",
    )

    simulation_config = SimulationConfig(
        name="Setup_bis",
        min_freq_ghz=4.0,
        n_modes=3,
        max_delta_f=0.1,
        max_passes=2,
    )

    report_config = ReportConfig(
        name="Fields",
        field_configs=[{"object_name": "main", "QuantityName": 'Mag_E', 'PlotFolder': 'E Field'}, {"object_name": "main", "QuantityName": 'Mag_H', 'PlotFolder': 'H Field'}]
    )

    ansys = ANSYS(configs=[render_config, simulation_config, report_config], run_upon_init=True)

In [1]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
design = designs.DesignPlanar()
gui = MetalGUI(design=design)

In [ ]:
# -*- coding: utf-8 -*-

from typing import List
import numpy as np
from qiskit_metal import Dict
from qiskit_metal.qlibrary.core import QRoute, QRoutePoint
from qiskit_metal.toolbox_metal.parsing import is_true

class BraggMirror(QRoute):
    """A CPW Bragg mirror with alternating sections of high and low impedance.
    
    Inherits QRoute class.
    
    QRoute Default Options:
        * pin_inputs: Dict
            * start_pin: Dict -- Component and pin string pair. Define which pin to start from
                * component: '' -- Name of component to start from, which has a pin
                * pin: '' -- Name of pin used for pin_start
            * end_pin=Dict -- Component and pin string pair. Define which pin to start from
                * component: '' -- Name of component to end on, which has a pin
                * pin: '' -- Name of pin used for pin_end
        * fillet: '0'
    
    Default Options:
        * n_sections: '5' -- Number of alternating impedance sections
        * section_length: '100um' -- Length of each section
        * center_width: '10um' -- Width of center conductor
        * gap_ratio: '2.0' -- Ratio between gap and center conductor width
        * impedance_ratio: '2.0' -- Ratio between high and low impedance sections
    """

    default_options = Dict(
        n_sections='5',
        section_length='100um',
        center_width='10um', 
        gap_ratio='2.0',
        impedance_ratio='2.0'
    )
    """Default options"""

    def make(self):
        """Generate the Bragg mirror geometry."""
        p = self.parse_options()

        # Get the basic parameters
        n_sections = int(p.n_sections)
        section_length = float(p.section_length)
        center_width = float(p.center_width)
        gap_ratio = float(p.gap_ratio)
        impedance_ratio = float(p.impedance_ratio)

        # Set up start and end pins
        self.set_pin("start")
        self.set_pin("end")

        # Get start and end points
        start_point = self.set_lead("start")
        end_point = self.set_lead("end")

        # Calculate direction vector between start and end
        direction = end_point.position - start_point.position
        unit_vector = direction / np.linalg.norm(direction)
        
        # Generate points for each section
        points = []
        current_point = start_point.position
        
        # Add start point
        points.append(current_point)
        
        # Create alternating sections
        for i in range(n_sections):
            # Move to next point
            current_point = current_point + unit_vector * section_length
            points.append(current_point)
            
            # Set width for this section
            if i % 2 == 0:
                # High impedance section
                self.options.trace_width = center_width
                self.options.gap = center_width * gap_ratio
            else:
                # Low impedance section
                self.options.trace_width = center_width * impedance_ratio
                self.options.gap = center_width * gap_ratio / impedance_ratio

        # Convert points to numpy array
        self.intermediate_pts = np.array(points)

        # Make the elements
        self.make_elements(self.get_points())

In [4]:
# Create the Bragg mirror
bragg = BraggMirror(
    design,
    name='bragg1',
    options=Dict(
        n_sections=5,
        section_length='1000um',
        center_width='10um',
        gap_ratio=2.0,
        impedance_ratio=2.0,
        pin_inputs=Dict(
            start_pin=Dict(component='TransmonCrossFL-321', pin='flux_line'),
            end_pin=Dict(component='TransmonCrossFL-321', pin='a')
        )
    )
)

In [5]:
design.rebuild()

In [6]:
gui.rebuild()
gui.show()